<a href="https://colab.research.google.com/github/Indhirayc/PBL_Kelompok1_LeafItToMe/blob/feature%2Fml-training/machine_learning/notebooks/02_data_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
#Melakukan Cleaning data
import os
from PIL import Image

dataset_path = '/content/drive/MyDrive/dataset'
corrupted_files = []

print("Memulai proses cleaning data...")

# Menelusuri setiap folder kelas di dalam dataset
for folder_name in os.listdir(dataset_path):
    folder_path = os.path.join(dataset_path, folder_name)

    # Pastikan yang dibaca adalah folder (bukan file nyasar)
    if os.path.isdir(folder_path):
        for file_name in os.listdir(folder_path):
            file_path = os.path.join(folder_path, file_name)

            try:
                # Mencoba membuka gambar dan memverifikasi integritasnya
                img = Image.open(file_path)
                img.verify()
            except (IOError, SyntaxError) as e:
                print(f"Gambar rusak ditemukan: {file_path}")
                corrupted_files.append(file_path)

                # Menghapus file yang rusak (hapus tanda '#' di bawah ini jika ingin otomatis dihapus)
                os.remove(file_path)

if len(corrupted_files) == 0:
    print("Cleaning selesai! Tidak ada gambar yang rusak. Dataset Anda aman.")
else:
    print(f"Cleaning selesai! Total {len(corrupted_files)} gambar rusak telah dihapus.")

Memulai proses cleaning data...
Cleaning selesai! Tidak ada gambar yang rusak. Dataset Anda aman.


In [8]:
import tensorflow as tf

dataset_path = '/content/drive/MyDrive/dataset'
batch_size = 32
img_size = (244, 244)

# Memuat dataset gambar dari direktori
train_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,#20% untuk validasi
    subset="training",
    seed=123, #Digunakan untuk pemisah antar data agar konsisten
    image_size=img_size,
    batch_size=batch_size
)

# Memuat data Validasi
val_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",
    seed = 123,
    image_size=img_size,
    batch_size=batch_size
)

#Layer Normalisasi (mengubah piksel dari 1-255 menjadi 0-1)
normalization_layer = tf.keras.layers.Rescaling(1./255)

#Menambahkan layer normalisasi ke dataset
train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))

#optimasi peforma
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

print("Preprocessing Selesai. Dataset Siap dilatih!")

Found 13781 files belonging to 5 classes.
Using 11025 files for training.
Found 13781 files belonging to 5 classes.
Using 2756 files for validation.
Preprocessing Selesai. Dataset Siap dilatih!
